# Create Cross-Validation Folds

This notebook creates a static, reusable cross-validation file so all models are trained on the **exact same splits**.

**Configuration**:
- `n_splits = 5` (ensures ~30 TDEs per fold from ~148 total)
- `shuffle = True`
- `random_state = 15` (fixed seed for reproducibility)

In [ ]:
# ==========================================
# CONFIGURATION & REPRODUCIBILITY
# ==========================================
import sys
from pathlib import Path

PROJECT_ROOT = Path().absolute().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from config_loader import load_config, get_path, get_seed, get_cv_config

config = load_config()
cv_config = get_cv_config()

N_SPLITS = cv_config['n_splits']
SHUFFLE = cv_config['shuffle']
RANDOM_STATE = get_seed()

USE_SYNTHETIC_DATA = True

DATA_DIR = get_path('data_processed')
INPUT_PATH = DATA_DIR / 'advanced_train_features.parquet'
OUTPUT_PATH = DATA_DIR / 'train_folds.csv'

if USE_SYNTHETIC_DATA:
    INPUT_PATH = DATA_DIR / 'synthetic_train_features.parquet'
    OUTPUT_PATH = DATA_DIR / 'synthetic_train_folds.csv'

print(f'Project Root: {PROJECT_ROOT}')
print(f'Random State: {RANDOM_STATE}')
print(f'N Splits: {N_SPLITS}')

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
import warnings

warnings.filterwarnings('ignore')

In [ ]:
print("Loading training data...")
df = pd.read_parquet(INPUT_PATH)
print(f"Data shape: {df.shape}")
print(f"\nClass distribution:")
print(df['target'].value_counts())

In [ ]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=SHUFFLE, random_state=RANDOM_STATE)

df['kfold'] = -1

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(df, df['target'])):
    df.loc[val_idx, 'kfold'] = fold_idx

print(f"Created {N_SPLITS} folds.")

In [ ]:
print("\n=== Sanity Check: Target=1 (TDE) count per fold ===")
print("Expected: ~29-30 TDEs per fold\n")

for fold in range(N_SPLITS):
    fold_data = df[df['kfold'] == fold]
    tde_count = fold_data['target'].sum()
    total_count = len(fold_data)
    print(f"Fold {fold}: {tde_count} TDEs / {total_count} total samples ({tde_count/total_count*100:.2f}%)")

print(f"\nTotal TDEs: {df['target'].sum()}")

In [ ]:
output_df = df[['object_id', 'target', 'kfold']]

output_df.to_csv(OUTPUT_PATH, index=False)
print(f"\nFolds saved to: {OUTPUT_PATH}")
print(f"Columns: {output_df.columns.tolist()}")
print(f"Shape: {output_df.shape}")

In [ ]:
print("\n=== Verification ===")
saved_df = pd.read_csv(OUTPUT_PATH)
print(f"File loaded successfully: {OUTPUT_PATH}")
print(f"Columns: {saved_df.columns.tolist()}")
print(f"Unique fold values: {sorted(saved_df['kfold'].unique())}")
print(f"\nFirst few rows:")
print(saved_df.head(10))